# TP — Première application d’un réseau neuronal

## Objectif

Dans ce TP, nous allons appliquer pas à pas un réseau neuronal à un problème de classification volontairement simplifié.

L'objectif n'est **pas de coder un réseau neuronal**, mais de comprendre les différentes étapes nécessaires pour passer :

```text
Données
   ↓
Préparation
   ↓
Construction du réseau
   ↓
Entraînement
   ↓
Évaluation
   ↓
Prédiction
```

À la fin du TP, vous devrez être capables d'expliquer ce que fait chacune de ces étapes.

# Contexte

Imaginons que nous disposions d'observations décrites par deux mesures numériques.

Chaque observation appartient à l'une des deux catégories :

- **Classe 0**
- **Classe 1**

Notre objectif est de construire un modèle capable de prédire la classe d'une nouvelle observation.

Pour simplifier volontairement le problème, les données seront artificielles.

> **Important :** dans un véritable projet, les données devraient provenir d'un problème métier réel et être analysées beaucoup plus précisément.

# Niveau 1 (Guidé)

## Étape 1 — Importer les bibliothèques

Nous allons utiliser :

- NumPy pour manipuler les données ;
- Pandas pour examiner les données ;
- Matplotlib pour les visualisations ;
- scikit-learn pour préparer et évaluer les données ;
- Keras pour construire le réseau neuronal.

Exécutez la cellule suivante.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Étape 2 — Générer les données

Pour cette démonstration, nous allons créer artificiellement notre dataset.

In [ ]:
X, y = make_moons(
    n_samples=500,
    noise=0.15,
    random_state=42
)

Nous obtenons :

- `X` : les variables explicatives ;
- `y` : la classe attendue.

Observons les dimensions.

In [ ]:
print(f'Nombre d observations : {len(X)}')
print(f'Nombre de variables : {X.shape[1]}')

Nous avons ici :

```text
500 observations
2 variables
```

# Étape 3 — Visualiser les données

Avant de construire un modèle, il est important de regarder les données.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X[y == 0, 0],
    X[y == 0, 1],
    label='Classe 0'
)

plt.scatter(
    X[y == 1, 0],
    X[y == 1, 1],
    label='Classe 1'
)

plt.xlabel('Variable 1')
plt.ylabel('Variable 2')
plt.title('Données du problème')
plt.legend()
plt.show()

### Observez le graphique

Les deux classes peuvent-elles être séparées simplement par une droite ?

Prenez quelques secondes pour observer la forme des données.

**Question :**

> Pourquoi un modèle qui ne pourrait construire qu'une frontière linéaire risque-t-il d'avoir des difficultés ici ?

# Étape 4 — Comprendre ce que nous voulons prédire

Notre dataset peut être représenté ainsi :

| Variable 1 | Variable 2 | Classe |
| ---------: | ---------: | -----: |
|        0.2 |        0.4 |      1 |
|       -0.8 |        0.5 |      0 |
|        1.1 |       -0.2 |      1 |
|       -0.4 |        0.8 |      0 |

Les deux premières colonnes sont les **entrées du réseau**.

La colonne `Classe` correspond à la **sortie attendue**.

Nous avons donc :

```text
X = [Variable 1, Variable 2]

y = Classe
```

# Étape 5 — Séparer les données

Nous ne devons pas utiliser toutes les données pour entraîner le modèle.

Nous allons conserver une partie des observations pour le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Observons les dimensions.

In [ ]:
print(f'Données d entraînement : {X_train.shape}')
print(f'Données de test : {X_test.shape}')

Nous avons maintenant deux ensembles :

```text
80 % → entraînement

20 % → test
```

### Question

Pourquoi ne pas simplement entraîner le modèle sur les 500 observations puis mesurer son accuracy sur ces mêmes 500 observations ?

# Étape 6 — Standardiser les données

Nos deux variables ont ici des échelles relativement proches.

Cependant, la standardisation est une étape courante avant l'utilisation d'un réseau neuronal.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Attention

Observez bien la différence entre :

In [ ]:
scaler.fit_transform(X_train)

et :

In [ ]:
scaler.transform(X_test)

Le scaler apprend ses paramètres uniquement à partir des données d'entraînement.

# Étape 7 — Construire le réseau neuronal

Nous allons maintenant créer notre premier réseau neuronal.

In [ ]:
model = Sequential([
    Dense(16, activation='relu', input_shape=(2,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

Observons son architecture.

In [ ]:
model.summary()

Nous avons :

```text
2 variables d'entrée
       ↓
16 neurones — ReLU
       ↓
8 neurones — ReLU
       ↓
1 neurone — Sigmoïde
```

### Question

Pourquoi avons-nous un seul neurone dans la couche de sortie ?

Parce que nous réalisons ici une **classification binaire**.

La sortie sera une valeur comprise entre 0 et 1.

# Étape 8 — Observer les paramètres du réseau

Le réseau contient des paramètres qui doivent être appris.

In [ ]:
model.count_params()

Le résultat correspond au nombre total de paramètres du réseau.

### Question

Ces paramètres sont-ils choisis manuellement ?

Non.

Ils seront ajustés pendant l'entraînement.

# Étape 9 — Configurer l'apprentissage

Avant de lancer l'entraînement, nous devons indiquer à Keras :

- comment mesurer l'erreur ;
- comment modifier les paramètres ;
- quelles métriques afficher.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

Nous retrouvons ici les notions vues dans le cours.

```text
Prédiction
    ↓
Loss
    ↓
Gradient
    ↓
Optimiseur
    ↓
Modification des poids
```

# Étape 10 — Entraîner le réseau

Nous pouvons maintenant lancer l'apprentissage.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

Le modèle réalise plusieurs passages sur les données.

Rappelez-vous :

> Une **epoch** correspond à un passage complet sur les données d'entraînement.

# Étape 11 — Observer l'apprentissage

Keras conserve l'historique de l'entraînement.

In [ ]:
history_df = pd.DataFrame(history.history)

history_df.head()

Nous allons maintenant visualiser l'accuracy.

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    history_df.index + 1,
    history_df['accuracy'],
    label='Entraînement'
)

plt.plot(
    history_df.index + 1,
    history_df['val_accuracy'],
    label='Validation'
)

plt.xlabel('Époque')
plt.ylabel('Accuracy')
plt.title('Évolution de l accuracy')
plt.legend()
plt.show()

### Observez

L'accuracy évolue-t-elle au cours de l'entraînement ?

Le réseau reste-t-il complètement stable ?

# Étape 12 — Observer la loss

Faisons la même chose avec la fonction de perte.

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    history_df.index + 1,
    history_df['loss'],
    label='Entraînement'
)

plt.plot(
    history_df.index + 1,
    history_df['val_loss'],
    label='Validation'
)

plt.xlabel('Époque')
plt.ylabel('Loss')
plt.title('Évolution de la loss')
plt.legend()
plt.show()

### Question

Quelle tendance générale devrait avoir la loss pendant l'apprentissage ?

En général, nous cherchons à la réduire.

# Étape 13 — Évaluer le modèle sur le test

Le jeu de test n'a pas servi à ajuster les paramètres du réseau.

Nous pouvons donc maintenant l'utiliser pour évaluer le modèle.

In [ ]:
loss, accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f'Loss sur le test : {loss:.4f}')
print(f'Accuracy sur le test : {accuracy:.2%}')

### Question

Pourquoi cette accuracy est-elle plus intéressante que l'accuracy obtenue sur les données d'entraînement ?

Parce que les observations de test n'ont pas été utilisées directement pour apprendre les paramètres du modèle.

# Étape 14 — Demander des prédictions

Le réseau ne retourne pas directement une classe.

Il retourne une valeur comprise entre 0 et 1.

In [ ]:
y_proba = model.predict(
    X_test_scaled,
    verbose=0
).ravel()

print(y_proba[:10])

Par exemple, nous pouvons obtenir :

```text
[0.03 0.97 0.12 0.91 ...]
```

Nous pouvons interpréter ces valeurs comme des scores ou probabilités associés à la classe positive dans le cadre de notre classification binaire.

# Étape 15 — Transformer les probabilités en classes

Nous allons utiliser `0.5` comme seuil.

In [ ]:
y_pred = (y_proba >= 0.5).astype(int)

print(y_pred[:10])

La logique est :

```text
probabilité < 0.5
        ↓
    Classe 0

probabilité ≥ 0.5
        ↓
    Classe 1
```

### Important

Le réseau neuronal produit une sortie.

La règle permettant de transformer cette sortie en décision est une étape distincte.

# Étape 16 — Examiner les erreurs

Calculons la matrice de confusion.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

Visualisons-la.

In [ ]:
plt.figure(figsize=(5, 4))

plt.imshow(cm)

plt.title('Matrice de confusion')
plt.xlabel('Prédiction')
plt.ylabel('Vérité')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i, j],
            ha='center',
            va='center'
        )

plt.xticks([0, 1])
plt.yticks([0, 1])

plt.show()

Nous pouvons maintenant voir :

```text
             Prédiction
             0       1

Réel  0      ?       ?

      1      ?       ?
```

### Question

Combien d'observations de la classe 0 ont été correctement prédites ?

Combien d'observations de la classe 1 ont été correctement prédites ?

# Étape 17 — Visualiser les prédictions

Nous pouvons revenir au graphique original et afficher les prédictions du modèle.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_test[y_pred == 0, 0],
    X_test[y_pred == 0, 1],
    label='Prédit : classe 0'
)

plt.scatter(
    X_test[y_pred == 1, 0],
    X_test[y_pred == 1, 1],
    label='Prédit : classe 1'
)

plt.xlabel('Variable 1')
plt.ylabel('Variable 2')
plt.title('Prédictions sur les données de test')
plt.legend()
plt.show()

### Question

Les prédictions semblent-elles suivre la structure des données ?

# Étape 18 — Tester une nouvelle observation

Imaginons maintenant que nous recevions une nouvelle observation.

In [ ]:
new_observation = np.array([
    [0.5, 0.8]
])

Cette observation doit subir la même préparation que les données d'entraînement.

In [ ]:
new_observation_scaled = scaler.transform(
    new_observation
)

Nous pouvons maintenant demander une prédiction.

In [ ]:
new_probability = model.predict(
    new_observation_scaled,
    verbose=0
)[0, 0]

print(
    f'Probabilité prédite : '
    f'{new_probability:.2%}'
)

Puis appliquer notre seuil :

In [ ]:
if new_probability >= 0.5:
    print('Classe prédite : 1')
else:
    print('Classe prédite : 0')

# Étape 19 — Reconstituer toute la chaîne

Nous venons de réaliser les étapes suivantes :

```mermaid
flowchart LR
    A["Dataset"] --> B["Train / Test"]
    B --> C["Standardisation"]
    C --> D["Réseau neuronal"]
    D --> E["Entraînement"]
    E --> F["Évaluation"]
    F --> G["Prédiction"]
```

Mais l'entraînement lui-même fonctionne selon :

```mermaid
flowchart LR
    A["Entrées"] --> B["Propagation avant"]
    B --> C["Prédiction"]
    C --> D["Loss"]
    D --> E["Backpropagation"]
    E --> F["Mise à jour des poids"]
    F --> B
```

# Niveau 2 (Réflexion)

## Étape 20 — Que se passerait-il si nous changions l'architecture ?

Nous avons utilisé :

```text
16 neurones
    ↓
8 neurones
    ↓
1 neurone
```

Que se passerait-il si nous utilisions seulement :

```text
2 neurones
    ↓
1 neurone
```

Modifiez uniquement cette cellule :

In [ ]:
model_small = Sequential([
    Dense(2, activation='relu', input_shape=(2,)),
    Dense(1, activation='sigmoid')
])

model_small.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_small = model_small.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

Évaluez le modèle.

In [ ]:
loss_small, accuracy_small = model_small.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f'Accuracy : {accuracy_small:.2%}')

### Questions

1. La performance est-elle comparable ?
2. Le réseau est-il plus ou moins complexe ?
3. Est-ce qu'un réseau plus grand est nécessairement meilleur ?

# Niveau 3 (Challenge)

## Étape 21 — Modifier le nombre de neurones

Revenez à l'architecture initiale.

Testez maintenant une architecture différente :

```text
2 entrées
   ↓
32 neurones — ReLU
   ↓
16 neurones — ReLU
   ↓
1 neurone — Sigmoïde
```

Construisez le modèle en modifiant uniquement l'architecture.

In [ ]:
model_large = Sequential([
    Dense(32, activation='relu', input_shape=(2,)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

Compilez-le et entraînez-le.

Comparez ensuite son comportement avec le modèle précédent.

### Questions

- Le nombre de paramètres a-t-il augmenté ?
- L'accuracy change-t-elle ?
- Le temps d'entraînement change-t-il ?
- Une architecture plus complexe apporte-t-elle toujours un gain important ?

# Étape 22 — Modifier le seuil de décision

Nous avons utilisé :

In [ ]:
threshold = 0.5

Testez maintenant plusieurs seuils :

In [ ]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    predictions = (
        y_proba >= threshold
    ).astype(int)

    accuracy_threshold = accuracy_score(
        y_test,
        predictions
    )

    print(
        f'Seuil = {threshold:.1f} '
        f'→ accuracy = {accuracy_threshold:.2%}'
    )

### Question

L'accuracy reste-t-elle identique lorsque le seuil change ?

# Bilan du TP

Nous avons appliqué un réseau neuronal de bout en bout.

La chaîne complète est :

```text
1. Comprendre les données
          ↓
2. Séparer train / test
          ↓
3. Préparer les données
          ↓
4. Définir l'architecture
          ↓
5. Compiler le modèle
          ↓
6. Entraîner
          ↓
7. Observer l'apprentissage
          ↓
8. Évaluer sur le test
          ↓
9. Analyser les erreurs
          ↓
10. Faire des prédictions
```

## Les trois idées essentielles

### 1. Le réseau apprend ses paramètres

Les poids et les biais sont ajustés pendant l'entraînement.

### 2. L'apprentissage repose sur une boucle

```text
Prédiction
    ↓
Loss
    ↓
Gradient
    ↓
Mise à jour
    ↓
Prédiction suivante
```

### 3. La performance ne se résume pas à l'entraînement

Nous devons vérifier que le modèle généralise sur des données qu'il n'a pas utilisées pour apprendre.

# Question finale

Sans regarder le code, essayez maintenant d'expliquer oralement le parcours d'une nouvelle observation :

```text
[Variable 1, Variable 2]
            ↓
            ?
            ↓
            ?
            ↓
            ?
            ↓
     Classe prédite
```

**Si vous êtes capables de raconter ce parcours avec vos propres mots, vous avez compris l'essentiel du fonctionnement d'un réseau neuronal.**